# Integer-LIF quantization distortion: statistical audit

This notebook reruns the frozen aggregate analysis and exposes the main audit tables. The statistical unit is the original UDD source image (35 clusters), not the 8,478 overlapping patches.

In [1]:
from pathlib import Path
import subprocess, sys
analysis_dir = Path.cwd()
if not (analysis_dir / 'build_analysis.py').exists():
    analysis_dir = Path('quantization_distortion_results/full_20260805/statistical_analysis')
subprocess.run([sys.executable, str(analysis_dir / 'build_analysis.py'), '--no-notebook'], check=True)

CompletedProcess(args=['/root/miniconda3/bin/python', '/root/autodl-tmp/LETNet/quantization_distortion_results/full_20260805/statistical_analysis/build_analysis.py', '--no-notebook'], returncode=0)

In [2]:
import json, pandas as pd
summary = json.loads((analysis_dir / 'analysis_summary.json').read_text())
pd.DataFrame(summary['representation'])

,mode,mode_label,source_count,layer_count,macro_disagreement,disagreement_ci_low,disagreement_ci_high,source_std_disagreement,source_cv_disagreement,micro_disagreement,...,macro_zero_reference,macro_zero_variant,macro_zero_delta,zero_delta_ci_low,zero_delta_ci_high,macro_saturation_reference,macro_saturation_variant,macro_saturation_delta,saturation_delta_ci_low,saturation_delta_ci_high
0,w4_qif,W4,35,78,0.108635,0.105637,0.111788,0.009501,0.087454,0.118299,...,0.904241,0.900505,-0.003736,-0.005982,-0.001581,0.000087,0.000024,-0.000063,-7.347670e-05,-0.000052
1,a4_input_qif,A4-input,35,78,0.087091,0.082885,0.091266,0.012807,0.147049,0.092214,...,0.904241,0.901232,-0.003009,-0.003553,-0.002469,0.000087,0.000092,0.000005,5.486000e-07,0.000011
2,w4a4_input_qif,W4A4-input,35,78,0.113808,0.110664,0.116935,0.009633,0.084642,0.131377,...,0.904241,0.899421,-0.004820,-0.007075,-0.002665,0.000087,0.000024,-0.000063,-7.300420e-05,-0.000053


In [3]:
stage = pd.read_csv(analysis_dir / 'stage_statistics.csv')
stage.query("mode == 'w4a4_input_qif'")[['stage_label','disagreement','signed_error','zero_delta','saturation_delta','distortion_contribution_share']]

,stage_label,disagreement,signed_error,zero_delta,saturation_delta,distortion_contribution_share
14,Stem,0.122898,0.031685,-0.024154,1.996526e-06,0.041533
15,Encoder 1,0.111678,0.006617,-0.007504,-1.952459e-06,0.150967
16,Encoder 2,0.047319,0.004495,-0.004456,-4.745122e-09,0.063966
17,Encoder 3,0.093058,0.010553,-0.011754,-2.848639e-06,0.125797
18,Decoder 4,0.043142,0.005459,-0.005012,-1.035301e-07,0.063180
19,Decoder 5,0.138519,0.017241,-0.014010,-2.377207e-05,0.202855
20,Decoder 6,0.240159,-0.058325,0.017566,-3.477355e-04,0.351702


In [4]:
layers = pd.read_csv(analysis_dir / 'layer_statistics.csv')
layers.sort_values('d_w4a4', ascending=False).head(12)[['layer','stage','d_w4a4','delta_d_full_minus_w4','signed_w4a4','zero_delta_w4a4']]

,layer,stage,d_w4a4,delta_d_full_minus_w4,signed_w4a4,zero_delta_w4a4
55,DAB_Block_6.DAB_Module_6_0.bn_relu.acti,decoder_6,0.425619,0.025508,-0.086166,0.025241
77,upsample_3.relu,decoder_6,0.342947,0.020530,-0.052025,0.007434
59,DAB_Block_6.DAB_Module_6_0.conv1x1_in.bn_prelu...,decoder_6,0.322535,0.014535,-0.004515,-0.003974
61,DAB_Block_6.DAB_Module_6_0.conv3x1.bn_prelu.acti,decoder_6,0.312739,0.009134,-0.030043,0.013217
60,DAB_Block_6.DAB_Module_6_0.conv1x3.bn_prelu.acti,decoder_6,0.296417,0.004790,-0.035754,0.015553
76,upsample_2.relu,decoder_5,0.291682,0.021017,0.010348,0.005203
58,DAB_Block_6.DAB_Module_6_0.conv1x1.bn_prelu.acti,decoder_6,0.288662,0.017392,-0.116677,0.060661
66,LC1.bn_prelu.acti,decoder_6,0.285154,0.019741,-0.006501,0.010580
44,DAB_Block_5.DAB_Module_5_0.bn_relu.acti,decoder_5,0.258824,0.019772,0.050678,-0.048637
62,DAB_Block_6.DAB_Module_6_0.dconv1x3.bn_prelu.acti,decoder_6,0.257576,0.004074,-0.058723,-0.012783


In [5]:
pd.read_csv(analysis_dir / 'correlation_statistics.csv')

,x,x_label,y,y_label,method,coefficient,p_value,bootstrap_ci_low,bootstrap_ci_high,source_count
0,d_w4a4,W4A4 code disagreement,prediction_flip_w4a4,prediction-flip rate,spearman,0.872829,8.189900e-12,0.764157,0.927498,35
1,d_w4a4,W4A4 code disagreement,prediction_flip_w4a4,prediction-flip rate,pearson,0.855835,5.684406e-11,0.777453,0.920679,35
2,d_w4a4,W4A4 code disagreement,miou_drop_w4a4,per-source mIoU drop,spearman,0.742577,3.268928e-07,0.509644,0.874441,35
3,d_w4a4,W4A4 code disagreement,miou_drop_w4a4,per-source mIoU drop,pearson,0.797416,9.807102e-09,0.641608,0.898955,35
4,mae_w4a4,W4A4 code MAE,prediction_flip_w4a4,prediction-flip rate,spearman,0.870588,1.073757e-11,0.744687,0.924313,35
5,mae_w4a4,W4A4 code MAE,prediction_flip_w4a4,prediction-flip rate,pearson,0.845872,1.583569e-10,0.762560,0.917328,35
6,mae_w4a4,W4A4 code MAE,miou_drop_w4a4,per-source mIoU drop,spearman,0.739496,3.878126e-07,0.489591,0.883050,35
7,mae_w4a4,W4A4 code MAE,miou_drop_w4a4,per-source mIoU drop,pearson,0.806181,5.071023e-09,0.655755,0.905615,35


## Interpretation boundary

These are frozen-checkpoint perturbation statistics. They establish that the selected checkpoint exhibits observable W4A4 code distortion and show where it is concentrated. They do not establish that QAD caused a repair; that requires matched trained QAT/KD/QAD checkpoints.